# LAB 1 for CS4243: Textures and Materials
## Notebook 2 — Material classification and defect localisation

> **Introduction.** This notebook runs Tasks A and B on real MVTec data. For
> Task B, normal models are fitted on training images, the pixel threshold is
> selected on public validation masks, and final results are reported on the
> held-out test split. Never tune a parameter after viewing test results.


## 1. Imports and reproducible configuration

Images are resized for a responsive walkthrough. Set `MAX_PER_GROUP = None`
when running the final experiment on every validation and test image.


In [ ]:
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

from texturelab.config import FeatureConfig, GaborConfig, NormalityConfig
from texturelab.data import load_mask, read_manifest
from texturelab.evaluation import evaluate_detection, evaluate_material
from texturelab.features import extract_local_features, global_pool
from texturelab.models import fit_material_classifier
from texturelab.normality import (
    fit_normal_model,
    predict_anomaly,
    select_mask_threshold,
)
from texturelab.supplied import load_image
from texturelab.visualize import plot_confusion, plot_reliability

MATERIALS = ("carpet", "grid", "leather", "tile", "wood")
IMAGE_SIZE = (64, 64)
MAX_PER_GROUP = 2  # Use None for the complete final evaluation.

feature_config = FeatureConfig(
    gabor=GaborConfig(
        frequencies=(0.10, 0.22),
        orientations=(0, np.pi / 2),
        kernel_size=11,
        pool_size=7,
    )
)
normal_config = NormalityConfig(
    image_percentile=99,
    threshold_percentile=99.5,
    score_pool_size=7,
    ignore_border=5,
)
mvtec = read_manifest(ROOT / "manifests" / "mvtec_textures.csv")

## 2. Audit the new split

The original labelled MVTec set is split deterministically within every
material/defect group: even-numbered files form validation and odd-numbered
files form test. This keeps every defect type in both partitions.


In [ ]:
split_counts = Counter(record.split for record in mvtec)
print(split_counts)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(split_counts.keys(), split_counts.values(), color=["#5b8ff9", "#61d9a5", "#f6bd16"])
ax.set(title="MVTec manifest split sizes", ylabel="images")
for index, value in enumerate(split_counts.values()):
    ax.text(index, value + 10, str(value), ha="center")
plt.show()

## 3. Task A — material classification

Task A uses only normal training images for fitting. The validation and test
sets below contain one good and one defective example per material so that the
walkthrough remains quick; use the complete sets for final reporting.


In [ ]:
def first_records(split, material, condition, count=10):
    # Select a deterministic small subset for notebook demonstrations.
    selected = [
        record
        for record in mvtec
        if record.split == split
        and record.material == material
        and (
            (condition == "good" and record.defect_type == "good")
            or (condition == "defective" and record.defect_type != "good")
        )
    ]
    return selected if count is None else selected[:count]


train_records = sum(
    ([r for r in mvtec if r.split == "train" and r.material == material][:4]
     for material in MATERIALS),
    [],
)
validation_records = sum(
    (first_records("validation", material, "good")
     + first_records("validation", material, "defective")
     for material in MATERIALS),
    [],
)
test_records = sum(
    (first_records("test", material, "good")
     + first_records("test", material, "defective")
     for material in MATERIALS),
    [],
)


def image_descriptor(record, config=feature_config):
    image = load_image(record.path, IMAGE_SIZE)
    feature_map, _ = extract_local_features(image, config)
    return global_pool(feature_map)


x_train = np.stack([image_descriptor(record) for record in train_records])
x_validation = np.stack([image_descriptor(record) for record in validation_records])
x_test = np.stack([image_descriptor(record) for record in test_records])

material_model = fit_material_classifier(
    x_train,
    [record.material for record in train_records],
    C=5.0,
)

### Visualise descriptor separation and test performance

PCA is used only for visualisation. The classifier itself receives the full
descriptor through a `StandardScaler` + `LogisticRegression` pipeline.


In [ ]:
all_descriptors = np.vstack([x_train, x_validation, x_test])
projection = PCA(n_components=2).fit_transform(all_descriptors)
groups = (
    [(record, "train") for record in train_records]
    + [(record, "validation") for record in validation_records]
    + [(record, "test") for record in test_records]
)

fig, ax = plt.subplots(figsize=(9, 6))
markers = {"train": "o", "validation": "s", "test": "X"}
for split in markers:
    for material in MATERIALS:
        chosen = np.array(
            [record.material == material and group_split == split
             for record, group_split in groups]
        )
        ax.scatter(
            projection[chosen, 0],
            projection[chosen, 1],
            marker=markers[split],
            label=f"{material} / {split}",
            alpha=0.8,
        )
ax.set(title="PCA of pooled handcrafted descriptors", xlabel="PC1", ylabel="PC2")
ax.legend(ncol=3, fontsize=7)
plt.show()

test_probabilities = material_model.predict_proba(x_test)
task_a_report = evaluate_material(
    [record.material for record in test_records],
    test_probabilities,
    material_model.classes_,
    ["good" if record.defect_type == "good" else "defective"
     for record in test_records],
)
print(task_a_report)
plot_confusion(
    np.asarray(task_a_report["overall"]["confusion_matrix"]),
    material_model.classes_,
)
plot_reliability(
    np.asarray([record.material for record in test_records]),
    test_probabilities,
    material_model.classes_,
)
plt.show()

## 4. Task B — fit normal patch models

One model is fitted per material using normal training patches. Local averaging
of the standardized score suppresses isolated responses, and an outer border
is ignored because reflected filtering makes it unreliable.


In [ ]:
normal_models = {}
for material in MATERIALS:
    records = [
        record
        for record in mvtec
        if record.split == "train" and record.material == material
    ]
    feature_maps = [
        extract_local_features(load_image(record.path, IMAGE_SIZE), feature_config)[0]
        for record in records
    ]
    normal_models[material] = fit_normal_model(feature_maps, normal_config)

## 5. Tune the mask threshold on validation only

This is the only stage allowed to inspect validation masks. The selected
threshold is copied into each material model and then frozen before test
evaluation.


In [ ]:
def evaluation_subset(split):
    # Keep each defect type represented while limiting notebook runtime.
    chosen = []
    for material in MATERIALS:
        groups = sorted({
            record.defect_type
            for record in mvtec
            if record.split == split and record.material == material
        })
        for defect_type in groups:
            records = [
                record
                for record in mvtec
                if record.split == split
                and record.material == material
                and record.defect_type == defect_type
            ]
            chosen.extend(records if MAX_PER_GROUP is None else records[:MAX_PER_GROUP])
    return chosen


def score_dataset(records, config, models):
    score_maps, image_scores, provisional_masks, true_masks = [], [], [], []
    for record in records:
        image = load_image(record.path, IMAGE_SIZE)
        feature_map, _ = extract_local_features(image, config)
        score_map, image_score, predicted_mask = predict_anomaly(
            feature_map,
            models[record.material],
            normal_config,
        )
        true_mask = (
            load_mask(record.mask_path, IMAGE_SIZE)
            if record.mask_path
            else np.zeros(IMAGE_SIZE[::-1], dtype=bool)
        )
        score_maps.append(score_map)
        image_scores.append(image_score)
        provisional_masks.append(predicted_mask)
        true_masks.append(true_mask)
    return score_maps, image_scores, provisional_masks, true_masks


task_b_validation = evaluation_subset("validation")
validation_score_maps, validation_image_scores, _, validation_masks = score_dataset(
    task_b_validation,
    feature_config,
    normal_models,
)
threshold_score_maps = validation_score_maps
threshold_masks = validation_masks
if normal_config.ignore_border:
    border = normal_config.ignore_border
    threshold_score_maps = [
        score[border:-border, border:-border]
        for score in validation_score_maps
    ]
    threshold_masks = [
        mask[border:-border, border:-border]
        for mask in validation_masks
    ]
selected_threshold = select_mask_threshold(
    threshold_score_maps,
    threshold_masks,
)
for model in normal_models.values():
    model.threshold = selected_threshold

print(f"Validation-selected pixel threshold: {selected_threshold:.4f}")

### Validation diagnostic — why the threshold plot looks difficult

This is intentionally not a clean two-cluster picture. Most pixels in a defective image are still normal background, while subtle defects may produce scores similar to ordinary texture variation. Resizing, local pooling, illumination changes, and the diagonal feature model further broaden both distributions. Consequently, the normal- and defect-pixel histograms overlap heavily and no vertical line can separate them perfectly.

> **Why the visual may look bad:** the logarithmic density axis exposes the long tails, and the enormous number of background pixels makes the normal distribution dominate. The selected line is an F1-optimal compromise on validation pixels—not evidence that the scores are well separated. Moving it left increases recall but creates many false positives; moving it right produces cleaner masks but misses weak or small defects. A single global threshold must also compromise across materials and defect types. Do not tune it again after viewing test results.


In [ ]:
validation_pixels = np.concatenate([score.ravel() for score in threshold_score_maps])
validation_labels = np.concatenate([mask.ravel() for mask in threshold_masks])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    validation_pixels[~validation_labels],
    bins=60,
    density=True,
    alpha=0.55,
    label="normal pixels",
)
ax.hist(
    validation_pixels[validation_labels],
    bins=60,
    density=True,
    alpha=0.65,
    label="defect pixels",
)
ax.axvline(selected_threshold, color="black", linestyle="--", label="chosen threshold")
ax.set(xlabel="smoothed standardized distance", ylabel="density",
       title="Validation-only threshold selection")
ax.set_yscale("log")
ax.legend()
ax.text(
    0.99,
    0.97,
    "Heavy overlap is expected\nNo threshold cleanly separates both groups",
    transform=ax.transAxes,
    ha="right",
    va="top",
    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "0.6"},
)
plt.show()

## 6. Freeze choices and report test results

The following metrics are the Task B results. Do not return to the validation
stage after inspecting them.


In [ ]:
task_b_test = evaluation_subset("test")
test_score_maps, test_image_scores, test_predicted_masks, test_true_masks = score_dataset(
    task_b_test,
    feature_config,
    normal_models,
)
task_b_report = evaluate_detection(
    [record.defect_type != "good" for record in task_b_test],
    test_image_scores,
    test_true_masks,
    test_predicted_masks,
    [record.defect_type for record in task_b_test],
)
print(task_b_report)

# Summarise test behaviour by defect type.
defect_names = [name for name in task_b_report["by_defect_type"] if name != "good"]
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(
    defect_names,
    [task_b_report["by_defect_type"][name]["f1"] for name in defect_names],
)
ax.set(title="Held-out test pixel F1 by defect type", ylabel="pixel F1", ylim=(0, 1))
ax.tick_params(axis="x", rotation=55)
plt.show()

### Inspect an easier colour defect and its patch scores

Colour defects are often more visible to this compact descriptor than small
structural defects. This example is illustrative: many MVTec samples will have
substantially weaker localization and should be discussed honestly.


In [ ]:
example_index = next(
    index
    for index, record in enumerate(task_b_test)
    if record.material == "carpet" and record.defect_type == "color"
)
example = task_b_test[example_index]
example_image = load_image(example.path, IMAGE_SIZE)
example_score = test_score_maps[example_index]
example_prediction = test_predicted_masks[example_index]
example_truth = test_true_masks[example_index]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
panels = [
    (example_image, "input", None),
    (example_score, "smoothed patch score", "inferno"),
    (example_prediction, "predicted mask", "gray"),
    (example_truth, "test ground truth", "gray"),
]
for ax, (data, title, cmap) in zip(axes, panels):
    ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
fig.suptitle("Held-out carpet colour defect")
fig.tight_layout()
plt.show()

## 7. Required feature-family ablation

Repeat the validation-threshold/test-report sequence for Gabor-only, edge-only,
and combined configurations. Do not share a threshold between branches unless
you justify doing so, and do not choose the winning branch from test results.


In [ ]:
ablation_configs = {
    "Gabor only": FeatureConfig(
        include_colour=False,
        include_gradient=False,
        include_edges=False,
        gabor=feature_config.gabor,
    ),
    "Edge only": FeatureConfig(
        include_colour=False,
        include_gabor=False,
        gabor=feature_config.gabor,
    ),
    "Combined": feature_config,
}

print("Configurations prepared:", list(ablation_configs))

In [ ]:
# YOUR CODE HERE
#
# TODO repeat the validation-to-test workflow for each configuration in ablation_configs

In [ ]:
# Save the frozen Task A/B artifacts for Notebook 4.
MODEL_DIR = ROOT / "outputs" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
task_ab_model_path = MODEL_DIR / "task_ab_models.joblib"
joblib.dump(
    {
        "material_classifier": material_model,
        "normal_models": normal_models,
        "feature_config": feature_config,
        "normality_config": normal_config,
        "materials": MATERIALS,
        "image_size": IMAGE_SIZE,
        "validation_selected_threshold": selected_threshold,
    },
    task_ab_model_path,
)
print("Saved frozen Task A/B models to:", task_ab_model_path)